In [1]:
!pip -q install -U chromadb pypdf openai python-dotenv pydantic sentence-transformers pandas pyarrow gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 603.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
import os, re, json, hashlib, asyncio, logging
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("ndi_capstone")

BASE = Path("/content/ndi_capstone")
BRONZE, SILVER, GOLD = BASE/"bronze", BASE/"silver", BASE/"gold"
for d in (BRONZE, SILVER, GOLD): d.mkdir(parents=True, exist_ok=True)

CHROMA_DIR = BASE/"chroma_db"
COLLECTION_NAME = "ndi_knowledge_base_v2"
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

print("Environment ready.")


Environment ready.


In [3]:
# ============================================================
# OpenRouter API Configuration
# ============================================================

import os
from getpass import getpass

os.environ["OPENROUTER_API_KEY"] = getpass("الصقي مفتاح OpenRouter هنا: ")
os.environ["OPENROUTER_MODEL"] = "openrouter/free"

print("API configured:", bool(os.getenv("OPENROUTER_API_KEY")))

الصقي مفتاح OpenRouter هنا: ··········
API configured: True


In [4]:
# Data Processing + Data Quality + Governance

@dataclass
class GovernanceMetadata:
    asset_id: str
    owner: str
    domain: str
    classification: str
    retention_days: int
    lineage: List[str]
    allowed_roles: List[str]


CATALOG = {}


def register_asset(asset_id, owner, domain, classification="Internal",
                   retention_days=365, lineage=None, allowed_roles=None):

    CATALOG[asset_id] = GovernanceMetadata(
        asset_id,
        owner,
        domain,
        classification,
        retention_days,
        lineage or [],
        allowed_roles or [
            "data_analyst",
            "governance_officer",
            "auditor"
        ]
    )


def access_check(asset_id, role):
    meta = CATALOG.get(asset_id)

    if not meta:
        return {
            "allowed": False,
            "reason": "Asset is not registered."
        }

    return {
        "allowed": role in meta.allowed_roles,
        "classification": meta.classification,
        "owner": meta.owner,
        "reason": "Allowed" if role in meta.allowed_roles else "Role denied"
    }


class DataQuality:

    required = [
        "event_id",
        "event_time",
        "asset_id",
        "value",
        "source"
    ]

    def run(self, df):

        missing = [
            c for c in self.required
            if c not in df.columns
        ]

        schema_ok = not missing

        if not schema_ok:
            return {
                "schema": {
                    "passed": False,
                    "missing_columns": missing
                },
                "overall_passed": False
            }

        completeness = (
            1 -
            df[self.required].isna().sum().sum()
            / (len(df) * len(self.required))
            if len(df) else 0
        )

        duplicates = int(
            df["event_id"].duplicated().sum()
        )

        times = pd.to_datetime(
            df["event_time"],
            errors="coerce",
            utc=True
        ).notna()

        values = pd.to_numeric(
            df["value"],
            errors="coerce"
        ).notna()

        sources = (
            df["source"]
            .astype(str)
            .str.len()
            .gt(0)
        )

        invalid = int(
            (~(times & values & sources)).sum()
        )

        result = {
            "schema": {
                "passed": True,
                "missing_columns": []
            },
            "completeness": {
                "passed": completeness >= .95,
                "score": round(float(completeness), 4)
            },
            "duplicates": {
                "passed": duplicates == 0,
                "count": duplicates
            },
            "validity": {
                "passed": invalid == 0,
                "invalid_count": invalid
            }
        }

        result["overall_passed"] = all(
            x["passed"] for x in result.values()
        )

        return result


def normalize_events(df):

    out = df.copy()

    out["event_time"] = pd.to_datetime(
        out["event_time"],
        errors="coerce",
        utc=True
    )

    out["value"] = pd.to_numeric(
        out["value"],
        errors="coerce"
    )

    out = out.dropna(
        subset=[
            "event_id",
            "event_time",
            "asset_id",
            "value",
            "source"
        ]
    )

    return out.drop_duplicates(
        "event_id",
        keep="last"
    )


dq = DataQuality()

register_asset(
    "NDI_KNOWLEDGE_BASE",
    "Data Governance Team",
    "Data Governance",
    classification="Internal",
    retention_days=3650,
    lineage=[
        "PDF",
        "extraction",
        "chunking",
        "embeddings",
        "ChromaDB"
    ]
)

print("Data Quality + Governance initialized.")

Data Quality + Governance initialized.


In [5]:
# Persistent vector knowledge base
CHROMA_DIR = BASE / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print("ChromaDB persistent storage ready.")
print(f"Storage path: {CHROMA_DIR}")

ChromaDB persistent storage ready.
Storage path: /content/ndi_capstone/chroma_db


In [6]:
# Vector database + robust PDF ingestion (bronze/silver/gold مفعّلة)

embedder=SentenceTransformer(EMBED_MODEL)
client=chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    collection=client.get_collection(COLLECTION_NAME)
except Exception:
    collection=client.create_collection(COLLECTION_NAME, metadata={"hnsw:space":"cosine"})

def clean_text(t):
    return re.sub(r"\s+"," ",t or "").strip()

def chunks(t, size=900, overlap=150):
    words=t.split()
    step=max(1,size-overlap)
    return [" ".join(words[i:i+size]) for i in range(0,len(words),step) if words[i:i+size]]

def ingest_pdf(pdf_path, start_page=1, end_page=None):
    pdf=Path(pdf_path)
    if not pdf.exists():
        raise FileNotFoundError(f"PDF not found: {pdf}")
    reader=PdfReader(str(pdf))
    end_page=min(end_page or len(reader.pages), len(reader.pages))

    # ---------- BRONZE: النص الخام كما استُخرج من الـPDF ----------
    bronze_records = []
    for page in range(start_page, end_page+1):
        raw_text = reader.pages[page-1].extract_text() or ""
        bronze_records.append({"source": pdf.name, "page": page, "raw_text": raw_text})
    bronze_path = BRONZE / f"{pdf.stem}_raw.jsonl"
    with open(bronze_path, "w", encoding="utf-8") as f:
        for r in bronze_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # ---------- SILVER: نص مُنظّف ومُقسّم إلى مقاطع (بدون embeddings بعد) ----------
    silver_records = []
    for r in bronze_records:
        text = clean_text(r["raw_text"])
        for n, ch in enumerate(chunks(text), 1):
            hid = hashlib.sha256(f"{r['source']}|{r['page']}|{n}|{ch}".encode()).hexdigest()[:20]
            silver_records.append({
                "id": f"NDI-{r['page']}-{n}-{hid}",
                "document": f"دليل نضيء — الصفحة {r['page']} — المقطع {n}\n{ch}",
                "source": r["source"], "page": r["page"], "chunk": n
            })
    if not silver_records:
        raise ValueError("No extractable text found.")
    silver_path = SILVER / f"{pdf.stem}_chunks.jsonl"
    with open(silver_path, "w", encoding="utf-8") as f:
        for r in silver_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # ---------- GOLD: المقاطع + متجهاتها + بيانات الحوكمة (جاهزة للاستخدام) ----------
    docs  = [r["document"] for r in silver_records]
    ids   = [r["id"] for r in silver_records]
    metas = [{"source": r["source"], "page": r["page"], "chunk": r["chunk"],
              "asset_id": "NDI_KNOWLEDGE_BASE"} for r in silver_records]

    vectors = embedder.encode(docs, normalize_embeddings=True, show_progress_bar=True).tolist()
    collection.upsert(ids=ids, documents=docs, embeddings=vectors, metadatas=metas)

    gold_path = GOLD / f"{pdf.stem}_indexed_manifest.json"
    with open(gold_path, "w", encoding="utf-8") as f:
        json.dump({"count": len(docs), "collection": COLLECTION_NAME,
                   "ids": ids, "asset_id": "NDI_KNOWLEDGE_BASE"}, f, ensure_ascii=False, indent=2)

    logger.info(f"Bronze: {bronze_path} | Silver: {silver_path} | Gold: {gold_path}")
    return len(docs)

print("ChromaDB ready. Records:", collection.count())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB ready. Records: 0


In [7]:
# ============================================================
# Index National Data Index PDF
# ============================================================

PDF_PATH = "/content/National-Data-Index_v1.0_AR.PDF"

print("الملف موجود:", os.path.exists(PDF_PATH))

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"لم يتم العثور على الملف: {PDF_PATH}"
    )

# فهرسة الوثيقة
ingest_pdf(PDF_PATH)

# التحقق
print("عدد الـchunks المفهرسة:", collection.count())

if collection.count() == 0:
    raise RuntimeError("❌ لم تتم فهرسة أي chunks من ملف NDI.")
else:
    print("✅ تمت فهرسة ملف National-Data-Index بنجاح.")

الملف موجود: True


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

عدد الـchunks المفهرسة: 227
✅ تمت فهرسة ملف National-Data-Index بنجاح.


In [8]:
# Advanced RAG: query expansion + hybrid retrieval + MMR

STOPWORDS={"من","في","على","إلى","عن","هذا","هذه","ما","هو","هي","و","أو","the","a","an","of","in","on","for","to","and","or"}

def tokenize(t):
    return [x for x in re.findall(r"[\w\u0600-\u06FF]+",t.lower()) if x not in STOPWORDS]

def lexical_score(q,d):
    a,b=set(tokenize(q)),set(tokenize(d))
    return len(a&b)/len(a) if a else 0

def expand_query(q):
    out=[q]
    for k,vals in {
        "جودة":["جودة البيانات","data quality"],
        "حوكمة":["حوكمة البيانات","data governance"],
        "امتثال":["الامتثال","compliance"],
        "مخاطر":["إدارة المخاطر","risk management"]
    }.items():
        if k in q: out += [q+" "+v for v in vals]
    return list(dict.fromkeys(out))[:5]

def cosine(a,b):
    a,b=np.asarray(a),np.asarray(b)
    den=np.linalg.norm(a)*np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den else 0

def mmr(qv, candidates, k=5, lam=.72):
    selected=[]; remaining=list(range(len(candidates)))
    while remaining and len(selected)<k:
        best=max(remaining,key=lambda i:
            lam*cosine(qv,candidates[i]["embedding"]) -
            (1-lam)*max([cosine(candidates[i]["embedding"],candidates[j]["embedding"]) for j in selected],default=0))
        selected.append(best); remaining.remove(best)
    return [candidates[i] for i in selected]

def retrieve(query, top_k=5, candidate_k=20):
    if collection.count()==0: return []
    pool={}
    for q in expand_query(query):
        qv=embedder.encode(q,normalize_embeddings=True).tolist()
        r=collection.query(query_embeddings=[qv],n_results=min(candidate_k,collection.count()),
                           include=["documents","metadatas","embeddings","distances"])
        for doc,meta,emb,dist in zip(r["documents"][0],r["metadatas"][0],r["embeddings"][0],r["distances"][0]):
            key=f'{meta.get("page")}|{meta.get("chunk")}'
            item=pool.setdefault(key,{"document":doc,"metadata":meta,"embedding":emb,"vector":0})
            item["vector"]=max(item["vector"],1-float(dist))
    candidates=list(pool.values())
    for x in candidates:
        x["lexical"]=lexical_score(query,x["document"])
        x["hybrid"]=.65*x["vector"]+.35*x["lexical"]
    candidates=sorted(candidates,key=lambda x:x["hybrid"],reverse=True)[:candidate_k]
    qv=embedder.encode(query,normalize_embeddings=True)
    return mmr(qv,candidates,min(top_k,len(candidates)))

print("Advanced RAG ready.")


Advanced RAG ready.


In [9]:
# Grounded generation + safe fallback

RAG_SYSTEM = '''You are a data-governance assistant.
Use only the supplied context. If the context is insufficient, say:
"لا أستطيع تأكيد ذلك من المصادر المفهرسة."
Cite important claims as [المصدر: صفحة X].
Do not claim that the answer is an official assessment or decision.'''

def answer(question, top_k=5):
    retrieved=retrieve(question,top_k)
    if not retrieved:
        return {"answer":"لا توجد مصادر مفهرسة كافية.","sources":[]}

    context="\n\n---\n\n".join(
        f"[المصدر: صفحة {x['metadata'].get('page','?')}]\n{x['document']}" for x in retrieved
    )
    api_key=os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        return {
            "answer":"لم يتم ضبط OPENROUTER_API_KEY؛ لذلك لن أختلق إجابة. هذه مقتطفات الأدلة المسترجعة:\n\n"+
                     "\n\n".join(f"[صفحة {x['metadata'].get('page')}] {x['document'][:700]}..." for x in retrieved),
            "sources":[x["metadata"] for x in retrieved]
        }

    from openai import OpenAI
    llm=OpenAI(base_url="https://openrouter.ai/api/v1",api_key=api_key)
    model=os.getenv("OPENROUTER_MODEL","openrouter/free")
    prompt=f'''السؤال:
{question}

السياق المسموح:
{context}

أجب بالعربية فقط، واربط الاستنتاجات بالمصادر.'''
    try:
        res=llm.chat.completions.create(
            model=model,
            messages=[{"role":"system","content":RAG_SYSTEM},{"role":"user","content":prompt}],
            temperature=.1
        )
        return {"answer":res.choices[0].message.content or "No answer.","sources":[x["metadata"] for x in retrieved]}
    except Exception as e:
        # Safe fallback when the selected model is unavailable, rate-limited, or the free quota is exhausted.
        return {
            "answer":"تعذر تشغيل نموذج اللغة حاليًا (قد يكون النموذج غير متاح أو تم تجاوز الحد المجاني). لن أختلق إجابة. هذه الأدلة المسترجعة من قاعدة المعرفة:\n\n" +
                     "\n\n".join(f"[صفحة {x['metadata'].get('page')}] {x['document'][:700]}..." for x in retrieved),
            "sources":[x["metadata"] for x in retrieved],
            "llm_error":str(e)[:300]
        }


In [10]:
# Real-time pipeline (مربوط فعليًا بـ ChromaDB)

class RealTimePipeline:
    def __init__(self):
        self.queue = asyncio.Queue()

    async def ingest(self, event):
        await self.queue.put(event)  # real-time ingestion point

    async def process_one(self):
        event = await self.queue.get()

        try:
            raw = pd.DataFrame([event])

            # Data quality validation
            quality = dq.run(raw)

            if not quality["schema"]["passed"]:
                return {"status": "rejected", "quality": quality}

            # Normalize and clean the event
            clean = normalize_events(raw)

            if clean.empty:
                return {"status": "rejected", "quality": quality,
                        "reason": "No valid records after normalization."}

            # ---------- ربط حقيقي بقاعدة المتجهات ----------
            record = clean.iloc[0]
            text = f"حدث لحظي — {record['asset_id']} — القيمة: {record['value']} — المصدر: {record['source']}"
            vec = embedder.encode(text, normalize_embeddings=True).tolist()
            doc_id = f"RT-{record['event_id']}"
            collection.upsert(
                ids=[doc_id],
                documents=[text],
                embeddings=[vec],
                metadatas=[{
                    "source": record["source"],
                    "asset_id": record["asset_id"],
                    "chunk": 0,
                    "page": 0,
                    "type": "real_time_event"
                }]
            )

            return {
                "status": "processed",
                "quality": quality,
                "records_processed": len(clean),
                "indexed_id": doc_id,
                "data": clean.to_dict(orient="records")
            }

        finally:
            self.queue.task_done()


async def demo_realtime():
    p = RealTimePipeline()

    events = [
        {"event_id": "evt-1", "event_time": datetime.now(timezone.utc).isoformat(),
         "asset_id": "customer_data", "value": 98.5, "source": "CRM"},
        {"event_id": "evt-2", "event_time": datetime.now(timezone.utc).isoformat(),
         "asset_id": "sales_data", "value": 88.2, "source": "ERP"}
    ]

    for e in events:
        await p.ingest(e)

    return [await p.process_one() for _ in events]


# ---------- تشغيل فعلي (لا تتركه معلّقًا بتعليق) ----------
results = await demo_realtime()
print(json.dumps(results, ensure_ascii=False, indent=2, default=str))
print("عدد السجلات في ChromaDB بعد ضخ الأحداث اللحظية:", collection.count())

[
  {
    "status": "processed",
    "quality": {
      "schema": {
        "passed": true,
        "missing_columns": []
      },
      "completeness": {
        "passed": "True",
        "score": 1.0
      },
      "duplicates": {
        "passed": true,
        "count": 0
      },
      "validity": {
        "passed": true,
        "invalid_count": 0
      },
      "overall_passed": true
    },
    "records_processed": 1,
    "indexed_id": "RT-evt-1",
    "data": [
      {
        "event_id": "evt-1",
        "event_time": "2026-09-16 08:39:13.149172+00:00",
        "asset_id": "customer_data",
        "value": 98.5,
        "source": "CRM"
      }
    ]
  },
  {
    "status": "processed",
    "quality": {
      "schema": {
        "passed": true,
        "missing_columns": []
      },
      "completeness": {
        "passed": "True",
        "score": 1.0
      },
      "duplicates": {
        "passed": true,
        "count": 0
      },
      "validity": {
        "passed": true,
  

In [11]:
# End-to-end orchestration + self-test

def end_to_end(question, role="data_analyst"):
    auth = access_check("NDI_KNOWLEDGE_BASE", role)

    if not auth["allowed"]:
        return {
            "status": "denied",
            "reason": auth["reason"]
        }

    result = answer(question)

    result["governance"] = auth

    result["architecture"] = {
        "data_architecture": "Event ingestion + processing + persistent vector storage",
        "real_time_pipeline": "asyncio.Queue",
        "vector_database": "ChromaDB",
        "advanced_rag": "query expansion + hybrid retrieval + MMR + grounded generation",
        "data_quality": "schema + completeness + duplicates + validity",
        "governance": "catalog + owner + classification + retention + lineage + RBAC"
    }

    return result


# Self-test without PDF or API

test = pd.DataFrame([
    {
        "event_id": "x",
        "event_time": datetime.now(timezone.utc).isoformat(),
        "asset_id": "demo",
        "value": 10,
        "source": "demo"
    },
    {
        "event_id": "x",
        "event_time": datetime.now(timezone.utc).isoformat(),
        "asset_id": "demo",
        "value": 11,
        "source": "demo"
    }
])

report = dq.run(test)

assert report["duplicates"]["count"] == 1
assert len(normalize_events(test)) == 1
assert access_check("NDI_KNOWLEDGE_BASE", "data_analyst")["allowed"]
assert not access_check("NDI_KNOWLEDGE_BASE", "unknown_role")["allowed"]

print("SELF-TEST PASSED")
print(json.dumps(report, ensure_ascii=False, indent=2, default=str))

SELF-TEST PASSED
{
  "schema": {
    "passed": true,
    "missing_columns": []
  },
  "completeness": {
    "passed": "True",
    "score": 1.0
  },
  "duplicates": {
    "passed": false,
    "count": 1
  },
  "validity": {
    "passed": true,
    "invalid_count": 0
  },
  "overall_passed": false
}


In [12]:
# Interactive RAG Chat — Colab

import gradio as gr

def chat_fn(message, history):
    message=(message or "").strip()
    if not message:
        return "اكتب سؤالك أولًا."

    result=end_to_end(message, role="data_analyst")
    if result.get("status")=="denied":
        return f"تم رفض الوصول: {result.get('reason','غير معروف')}"

    answer_text=result.get("answer", "لا توجد إجابة.")
    sources=result.get("sources", [])

    if sources:
        source_lines=[]
        seen=set()
        for s in sources:
            key=(s.get("source"), s.get("page"), s.get("chunk"))
            if key in seen:
                continue
            seen.add(key)
            source_lines.append(
                f"- {s.get('source','غير معروف')} — الصفحة {s.get('page','?')} — المقطع {s.get('chunk','?')}"
            )
        answer_text += "\n\n**المصادر المسترجعة:**\n" + "\n".join(source_lines)

    return answer_text

chat = gr.ChatInterface(
    fn=chat_fn,
    title="💬 NDI Governance & RAG Chat",
    description="اسأل بالعربية. سيبحث النظام في قاعدة ChromaDB ثم يجيب من الأدلة المسترجعة فقط، مع إظهار المصادر. إذا لم يتوفر نموذج LLM أو انتهى الحد المجاني، سيعرض الأدلة بدل اختلاق إجابة.",
    textbox=gr.Textbox(placeholder="مثال: ما متطلبات حوكمة البيانات؟", container=True, scale=7),
    examples=[
        "ما متطلبات حوكمة البيانات؟",
        "ما متطلبات جودة البيانات؟",
        "ما الضوابط أو المتطلبات المتعلقة بالامتثال؟"
    ],
)

chat.launch(share=False, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>